# Autotune — Servidor Ollama Remoto com GPU (Kaggle v2)

Notebook limpo — instala tudo do zero em cada sessão.

## Pré-requisitos
- Acelerador GPU activado nas definições do notebook
- Token ngrok configurado na célula 5

## Arquitetura
```
Backend / RAG → URL ngrok → Ollama no Kaggle (GPU)
```

# 1 - Instalar dependências e Ollama

In [ ]:
!apt-get install -y zstd lshw pciutils -q
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pyngrok -q
!ls /usr/local/lib/ollama/

# 2 - Imports e funções auxiliares

In [ ]:
import os
import time
import subprocess
import requests
from datetime import datetime
from pyngrok import ngrok, conf

MODELS_DIR = '/kaggle/working/ollama_models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['OLLAMA_MODELS'] = MODELS_DIR

print('Modelos em:', MODELS_DIR)


def stop_ollama():
    print('A parar processos antigos do Ollama...')
    subprocess.run(['killall', 'ollama'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)
    print('Processos antigos terminados.')


def start_ollama():
    env = os.environ.copy()
    env['PATH'] += ':/usr/local/bin'
    env['OLLAMA_MODELS'] = MODELS_DIR
    env['OLLAMA_HOST'] = '0.0.0.0'
    env['OLLAMA_ORIGINS'] = '*'
    env['CUDA_VISIBLE_DEVICES'] = '0'
    env['OLLAMA_GPU_LAYERS'] = '999'

    print('A iniciar servidor Ollama...')
    process = subprocess.Popen(
        ['/usr/local/bin/ollama', 'serve'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=env
    )

    # Aguardar o Ollama iniciar
    for i in range(12):
        time.sleep(5)
        try:
            r = requests.get('http://localhost:11434/api/tags', timeout=3)
            if r.status_code == 200:
                print('Servidor Ollama iniciado.')
                return process
        except:
            print(f'A aguardar... ({(i+1)*5}s)')

    print('AVISO: Ollama demorou mais do esperado.')
    return process


def check_ollama_status():
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=10)
        if r.status_code == 200:
            print('Ollama está ativo.')
            return True
        print('Ollama respondeu com status:', r.status_code)
        return False
    except Exception as e:
        print('Erro ao contactar o Ollama:', e)
        return False


def pull_model(model_name):
    print(f'A descarregar modelo: {model_name}')
    result = subprocess.run(
        f'OLLAMA_MODELS={MODELS_DIR} ollama pull {model_name}',
        shell=True
    )
    if result.returncode == 0:
        print(f'Modelo {model_name} pronto.')
    else:
        print(f'Erro ao descarregar {model_name}.')


def list_models():
    print('Modelos disponíveis:')
    subprocess.run(f'OLLAMA_MODELS={MODELS_DIR} ollama list', shell=True)


def test_generation(model_name):
    print(f'A testar modelo: {model_name}')
    try:
        r = requests.post(
            'http://localhost:11434/api/chat',
            json={
                'model': model_name,
                'messages': [{'role': 'user', 'content': 'Responde apenas: OK'}],
                'stream': False
            },
            timeout=120
        )
        if r.status_code == 200:
            print('Resposta:', r.json().get('message', {}).get('content', ''))
            print(f'{model_name} está a funcionar.')
        else:
            print('Erro:', r.text[:200])
    except Exception as e:
        print('Erro ao testar:', e)


def start_ngrok_tunnel(ngrok_token, domain, port=11434):
    conf.get_default().auth_token = ngrok_token
    print('A limpar túneis antigos...')
    ngrok.kill()
    print('A criar túnel ngrok...')
    tunnel = ngrok.connect(port, domain=domain)
    print('=' * 60)
    print('OLLAMA_RAG_BASE_URL=' + tunnel.public_url)
    print('=' * 60)
    return tunnel


def keep_alive(interval_seconds=30):
    print('Servidor ativo. Keep-alive a correr...')
    print('-' * 40)
    while True:
        try:
            r = requests.get('http://localhost:11434/api/tags', timeout=5)
            models = [m['name'] for m in r.json().get('models', [])]
            print('[' + datetime.now().strftime('%H:%M:%S') + '] OK - ' + ', '.join(models))
        except Exception as e:
            print('[' + datetime.now().strftime('%H:%M:%S') + '] ERRO: ' + str(e))
        time.sleep(interval_seconds)

# 3 - Iniciar Ollama e verificar GPU

In [ ]:
stop_ollama()
ollama_process = start_ollama()
!nvidia-smi
check_ollama_status()

# 4 - Descarregar modelos

In [ ]:
pull_model('qwen2.5:14b')
list_models()

# 5 - Testar modelos

In [ ]:
test_generation('qwen2.5:14b')

# 6 - Configurar e iniciar ngrok

In [ ]:
NGROK_TOKEN = '38MUdfuiCncbmMq2OrR0l9PqNTH_6Q6G7ugANjh51tDbqc2ZZ'
MEU_DOMINIO = 'irenic-daniel-unshowily.ngrok-free.dev'

ngrok.kill()
tunnel = start_ngrok_tunnel(
    ngrok_token=NGROK_TOKEN,
    domain=MEU_DOMINIO,
    port=11434
)

# Keep-alive

In [ ]:
keep_alive(interval_seconds=30)